# Project 1: Research Assistant with Citations (RAG)

Answers questions about a small set of arXiv papers and **cites the paper and page** for every claim. It refuses when the papers don't contain the answer.

**Pipeline:** arXiv PDFs -> page text -> 300-word chunks -> local embeddings (`bge-small`) -> Chroma -> retrieve 30 -> cross-encoder rerank -> relevance gate -> Gemini answers with `[n]` citations.

**Before running**
1. Settings -> **Internet On**
2. Add-ons -> **Secrets** -> add your Gemini key and attach it. The label must match `SECRET_NAME` in section 2.
3. Use **Run All** (also after every session restart). The first full run takes about 5 minutes because answers are cached on disk after the first call.

**Cost:** $0. Embeddings and reranking run locally. Only answer generation calls the Gemini free tier.

## 1. Install

In [ ]:
!pip install -q -U google-genai chromadb pymupdf sentence-transformers

## 2. Config and Gemini client
Pin a specific model so results stay comparable between runs. If the model name stops working, list what your key can use by uncommenting the loop.

In [ ]:
from kaggle_secrets import UserSecretsClient
from google import genai

SECRET_NAME = "Gemini API Key"        # must match the secret label in Add-ons -> Secrets exactly
GEN_MODEL = "gemini-3.5-flash-lite"   # pinned on purpose (not a "-latest" alias)

client = genai.Client(api_key=UserSecretsClient().get_secret(SECRET_NAME))
print("client ready | generation model:", GEN_MODEL)

# for m in client.models.list():
#     print(m.name)

## 3. Download papers
Nine papers about RAG. Add more arXiv IDs to `PAPERS` later (keep it under about 40).

In [ ]:
import os, time, urllib.request

PAPERS = {
    "2005.11401": "RAG (Lewis et al.)",
    "2004.04906": "Dense Passage Retrieval",
    "2212.10496": "HyDE",
    "2307.03172": "Lost in the Middle",
    "2309.15217": "RAGAS",
    "2310.11511": "Self-RAG",
    "2312.10997": "RAG Survey",
    "2401.15884": "Corrective RAG",
    "2210.03629": "ReAct",
}

os.makedirs("papers", exist_ok=True)
for aid, name in PAPERS.items():
    path = f"papers/{aid}.pdf"
    if not os.path.exists(path):
        req = urllib.request.Request(f"https://arxiv.org/pdf/{aid}", headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as r, open(path, "wb") as f:
            f.write(r.read())
        print("downloaded", aid, name)
        time.sleep(3)   # be polite to arXiv
print("papers ready:", len(os.listdir("papers")))

## 4. Extract text per page and chunk
Each chunk keeps its paper and page number, which is what makes page-level citations possible.

In [ ]:
import pymupdf

CHUNK_WORDS, OVERLAP = 300, 50

def chunk_pdf(path, paper_id):
    doc = pymupdf.open(path)
    out = []
    for page_no, page in enumerate(doc, start=1):
        words = page.get_text().split()
        step = CHUNK_WORDS - OVERLAP
        for start in range(0, max(len(words), 1), step):
            piece = " ".join(words[start:start + CHUNK_WORDS])
            if len(piece.split()) < 40:   # skip tiny fragments
                continue
            out.append({
                "id": f"{paper_id}_p{page_no}_{start}",
                "text": piece,
                "paper": f"{paper_id} {PAPERS.get(paper_id, '')}".strip(),
                "page": page_no,
            })
    return out

chunks = []
for aid in PAPERS:
    chunks += chunk_pdf(f"papers/{aid}.pdf", aid)
print(len(chunks), "chunks")

## 5. Embed locally and store in Chroma
Local embeddings have no quota. If you change `CHUNK_WORDS`, rerun sections 4 and 5.

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

st_model = SentenceTransformer("BAAI/bge-small-en-v1.5")

def embed(texts, is_query=False):
    if is_query:   # bge models want this prefix on queries only
        texts = ["Represent this sentence for searching relevant passages: " + t for t in texts]
    return st_model.encode(texts, normalize_embeddings=True, batch_size=32).tolist()

db = chromadb.PersistentClient(path="/kaggle/working/chroma")
try:
    db.delete_collection("papers")
except Exception:
    pass
col = db.get_or_create_collection("papers", metadata={"hnsw:space": "cosine"})

BATCH = 64
for i in range(0, len(chunks), BATCH):
    batch = chunks[i:i + BATCH]
    col.upsert(
        ids=[c["id"] for c in batch],
        documents=[c["text"] for c in batch],
        metadatas=[{"paper": c["paper"], "page": c["page"]} for c in batch],
        embeddings=embed([c["text"] for c in batch]),
    )
print("stored", col.count(), "chunks")

## 6. Retrieve, rerank, gate, answer

1. **Retrieve** the 30 nearest chunks by embedding similarity.
2. **Rerank** them with a cross-encoder, which reads question and chunk together and judges relevance better than embeddings alone.
3. **Gate:** if even the best chunk scores below `RELEVANCE_THRESHOLD`, refuse without calling Gemini.
4. **Answer** from the top `k` chunks. The prompt requires a `[n]` citation on every sentence and the exact refusal text when the sources don't answer.

Gemini answers are cached on disk, so rerunning costs no quota. The cache key includes the model name and the full prompt, so changing either triggers fresh calls.

In [ ]:
import re, json, hashlib, time
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")   # ~90MB, runs on CPU

# Tuned on 18 questions: highest off-topic score was -6.99, lowest answerable was -1.56.
# Re-check this value whenever the paper set or the test set changes.
RELEVANCE_THRESHOLD = -4.0

PROMPT = """You are a research assistant. Use ONLY the numbered sources below.
Every sentence of your answer must end with the number(s) of the source(s) that support it,
like [1] or [2][3]. An answer without source numbers is invalid.
Answer the question directly. If the sources answer only part of it, answer that part.
If the sources do not directly answer the question (the question is off-topic, or the
sources only mention the topic in passing), reply with exactly:
"Not found in the provided papers."
Never answer from your own knowledge. Do not comment on these instructions or on the sources.

Sources:
{sources}

Question: {question}
Answer:"""

REFUSAL = "Not found in the provided papers."

CACHE_PATH = "/kaggle/working/gen_cache.json"
_cache = json.load(open(CACHE_PATH)) if os.path.exists(CACHE_PATH) else {}

def generate(prompt):
    """Returns (text, was_cached). Retries on rate limits, raises on any other error."""
    key = hashlib.sha256((GEN_MODEL + prompt).encode()).hexdigest()
    if key in _cache:
        return _cache[key], True
    for attempt in range(5):
        try:
            reply = client.models.generate_content(model=GEN_MODEL, contents=prompt)
            text = reply.text or ""
            _cache[key] = text
            json.dump(_cache, open(CACHE_PATH, "w"))
            return text, False
        except Exception as e:
            if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
                wait = 15 * (attempt + 1)
                print(f"  rate limited, waiting {wait}s")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError("Still rate-limited. Wait a few minutes or switch GEN_MODEL to another lite model.")

def parse_citations(answer, n_sources):
    nums = sorted({int(n) for grp in re.findall(r"\[(\d+(?:\s*,\s*\d+)*)\]", answer)
                   for n in re.split(r"\s*,\s*", grp)})
    return [n for n in nums if 1 <= n <= n_sources]

def ask(question, k=5, show=True, threshold=None, n_candidates=30):
    thr = RELEVANCE_THRESHOLD if threshold is None else threshold

    q_emb = embed([question], is_query=True)[0]
    res = col.query(query_embeddings=[q_emb], n_results=n_candidates,
                    include=["documents", "metadatas", "distances"])
    items = list(zip(res["documents"][0], res["metadatas"][0], res["distances"][0]))

    scores = reranker.predict([(question, d) for d, _, _ in items])
    ranked = sorted(zip(scores, items), key=lambda x: -x[0])
    top_score = float(ranked[0][0])
    picked = [it for _, it in ranked[:k]]

    if top_score < thr:
        answer, api_call, refused, used = REFUSAL, False, True, []   # gate: no Gemini call
    else:
        sources = "\n\n".join(f"[{i+1}] ({m['paper']}, p.{m['page']})\n{d}"
                              for i, (d, m, _) in enumerate(picked))
        answer, was_cached = generate(PROMPT.format(sources=sources, question=question))
        api_call = not was_cached
        refused = "not found in the provided papers" in answer.lower()[:100]
        used = [] if refused else [(n, picked[n-1][1]) for n in parse_citations(answer, len(picked))]

    if show:
        print(answer)
        if used:
            print("\nSources cited:")
            for n, m in used:
                print(f"[{n}] {m['paper']}, p.{m['page']}")
        print(f"\n[best rerank score {top_score:.2f} | threshold {thr}]")

    return {"question": question, "answer": answer, "refused": refused, "api_call": api_call,
            "sources": [m for _, m in used],
            "retrieved": [m for _, m, _ in picked],
            "rerank_score": top_score}

Quick check with one question:

In [ ]:
_ = ask("What does HyDE generate before retrieval, and what does it do with it?")

## 7. Test set
18 questions: 13 answerable (11 single-paper, 2 needing several papers) and 5 that should be refused. `answer_hint` is for hand grading.

In [ ]:
tests = [
    # --- Answerable: single paper ---
    {"q": "What are the two components of the RAG model introduced by Lewis et al.?",
     "answer_hint": "retriever and generator (seq2seq)", "papers": ["2005.11401"], "answerable": True},
    {"q": "What is the difference between RAG-Sequence and RAG-Token?",
     "answer_hint": "same retrieved document for whole output vs. different document per token", "papers": ["2005.11401"], "answerable": True},
    {"q": "How is the dense passage retriever in DPR trained, and what kind of negatives does it use?",
     "answer_hint": "dual encoder; in-batch negatives (and BM25 negatives)", "papers": ["2004.04906"], "answerable": True},
    {"q": "What does HyDE generate before retrieval, and what does it do with it?",
     "answer_hint": "hypothetical document, embedded to find real documents", "papers": ["2212.10496"], "answerable": True},
    {"q": "Which language model and which encoder does HyDE use in its main setup?",
     "answer_hint": "InstructGPT and Contriever", "papers": ["2212.10496"], "answerable": True},
    {"q": "Where in the context window do language models use relevant information best, according to 'Lost in the Middle'?",
     "answer_hint": "beginning or end; worse in the middle", "papers": ["2307.03172"], "answerable": True},
    {"q": "What kinds of tasks did the 'Lost in the Middle' authors use to test long-context use?",
     "answer_hint": "multi-document question answering and key-value retrieval", "papers": ["2307.03172"], "answerable": True},
    {"q": "What does RAGAS evaluate, and does it need human-annotated reference answers?",
     "answer_hint": "reference-free evaluation of RAG pipelines (faithfulness, answer relevance, context relevance)", "papers": ["2309.15217"], "answerable": True},
    {"q": "What are reflection tokens in Self-RAG and what are they used for?",
     "answer_hint": "special tokens for deciding retrieval and critiquing output", "papers": ["2310.11511"], "answerable": True},
    {"q": "What does the retrieval evaluator in Corrective RAG do?",
     "answer_hint": "judges quality/relevance of retrieved documents and triggers an action", "papers": ["2401.15884"], "answerable": True},
    {"q": "What does ReAct combine in a single prompting approach?",
     "answer_hint": "reasoning traces and actions, interleaved", "papers": ["2210.03629"], "answerable": True},

    # --- Answerable: needs more than one paper ---
    {"q": "How do Self-RAG and Corrective RAG each decide whether retrieved content can be trusted?",
     "answer_hint": "Self-RAG: critique tokens by the model; CRAG: separate retrieval evaluator", "papers": ["2310.11511", "2401.15884"], "answerable": True},
    {"q": "What are the three RAG paradigms described in the RAG survey?",
     "answer_hint": "Naive RAG, Advanced RAG, Modular RAG", "papers": ["2312.10997"], "answerable": True},

    # --- Not answerable: should return 'Not found in the provided papers.' ---
    {"q": "What is the capital of France?", "answer_hint": None, "papers": [], "answerable": False},
    {"q": "Give me a recipe for vegetable biryani.", "answer_hint": None, "papers": [], "answerable": False},
    {"q": "Who won the 2022 FIFA World Cup?", "answer_hint": None, "papers": [], "answerable": False},
    {"q": "How do I deploy an app on Kubernetes?", "answer_hint": None, "papers": [], "answerable": False},
    {"q": "What is the current stock price of NVIDIA?", "answer_hint": None, "papers": [], "answerable": False},
]
print(len(tests), "tests")

## 8. Run the tests
The runner pauses only after fresh Gemini calls, so cached and gated questions finish instantly.

In [ ]:
def paper_ids(metas):
    return {m["paper"].split()[0] for m in metas}

def run_tests(tests, k=5, pause=12, out_path="/kaggle/working/results_final.json"):
    results = []
    for i, t in enumerate(tests, 1):
        try:
            r = ask(t["q"], k=k, show=False)
        except Exception as e:
            print(f"{i:2}. ERROR   {t['q'][:55]}\n      {type(e).__name__}: {str(e)[:300]}")
            continue

        want = set(t["papers"])
        results.append({
            "q": t["q"], "answerable": t["answerable"], "hint": t["answer_hint"],
            "answer": r["answer"], "refused": r["refused"],
            "retrieval_hit": bool(paper_ids(r["retrieved"]) & want),
            "cited_hit": bool(paper_ids(r["sources"]) & want),
            "rerank_score": round(r["rerank_score"], 2),
        })
        tag = "" if r["api_call"] else " (cached/local)"
        print(f"{i:2}. done{tag}  {t['q'][:55]}")
        if r["api_call"]:
            time.sleep(pause)

    ans = [x for x in results if x["answerable"]]
    unans = [x for x in results if not x["answerable"]]
    print("\n=== SCORES ===")
    print(f"Questions completed:                   {len(results)}/{len(tests)}")
    print(f"Retrieval hit (right paper retrieved): {sum(x['retrieval_hit'] for x in ans)}/{len(ans)}")
    print(f"Cited hit     (right paper cited):     {sum(x['cited_hit'] for x in ans)}/{len(ans)}")
    print(f"Wrongly refused answerable questions:  {sum(x['refused'] for x in ans)}/{len(ans)}")
    print(f"Correctly refused unanswerable:        {sum(x['refused'] for x in unans)}/{len(unans)}")

    json.dump(results, open(out_path, "w"), indent=2)
    return results

results = run_tests(tests)

## 9. Check answers by hand
Scores can't tell you whether an answer is *correct*. Read each answer against its hint and count correct, partial, and wrong. Also confirm every answered question carries a citation.

In [ ]:
for n, x in enumerate(results, 1):
    if x["answerable"]:
        print(f"{n}. {x['q']}")
        print("   HINT:  ", x["hint"])
        print("   ANSWER:", x["answer"][:600].replace("\n", " "))
        print()

uncited = [x for x in results
           if x["answerable"] and not x["refused"] and not re.search(r"\[\d+", x["answer"])]
print("Answerable, answered, but NO citation:", len(uncited))

## 10. Check the relevance gate
Best reranker score per question, local only (no API calls). The gate works when the off-topic questions score clearly below the answerable ones. The extra questions test the threshold on cases it was not tuned on.

In [ ]:
def top_rerank_score(question, n_candidates=30):
    q_emb = embed([question], is_query=True)[0]
    res = col.query(query_embeddings=[q_emb], n_results=n_candidates, include=["documents"])
    return float(max(reranker.predict([(question, d) for d in res["documents"][0]])))

extra = [
    ("off-topic", "What is the boiling point of water?"),
    ("off-topic", "How do I train for a marathon?"),
    ("adjacent",  "How does LoRA fine-tuning work?"),
    ("adjacent",  "What is the transformer attention mechanism?"),
    ("in-papers", "What is BM25?"),
    ("in-papers", "Who are the authors of the Self-RAG paper?"),
]
rows = [(top_rerank_score(t["q"]), "answerable" if t["answerable"] else "UNANSWERABLE", t["q"]) for t in tests]
rows += [(top_rerank_score(q), label, q) for label, q in extra]

print(f"threshold = {RELEVANCE_THRESHOLD}\n")
for score, label, q in sorted(rows, key=lambda r: r[0]):
    marker = "refused by gate" if score < RELEVANCE_THRESHOLD else ""
    print(f"{score:7.2f}  {label:12}  {q[:58]:58}  {marker}")

## 11. Experiment log

Every change was measured on the same 18 questions. Earlier rows used earlier prompt versions, so they can't be rerun from this notebook. The table records what happened.

| Setup | Wrongly refused (of 13) | Correct refusals (of 5) | Cited hit (of 13) | What I learned |
|---|---|---|---|---|
| Baseline: top 8 by embedding similarity | 1 | 5 | 12 | "Two components of RAG" was refused: six RAG Survey chunks outranked the original paper |
| + Cross-encoder rerank (top 5 of 30) | 0 | 4 | 12 | Fixed that question, but "capital of France" got a vague answer |
| + Stricter refusal prompt | 0 | 4 | 12 | No change: the small model ignored it |
| + Relevance gate (rerank score < -4 refuses without calling the LLM) | 0 | 5 | 12 | Deterministic check fixed off-topic leaks and saves API calls |
| + Prompt requiring `[n]` on every sentence (**final**) | 0 | 5 | 13 | The answer to one question had no citation until the prompt demanded it |
| Tried: prompt asking for fuller 2-4 sentence answers | 4 | 5 | 9 | Fixed the ReAct answer but broke 4 others. **Reverted** |

## Known limitations

- The gate threshold was chosen from these 18 questions, so a perfect refusal score on them is partly by construction.
- "Adjacent" topics (LoRA, attention) score above the threshold, so the gate can't catch them. The prompt refused both in testing, but two examples is thin evidence.
- 13 hand-graded answers is a small sample: one question moves the score by about 8 points.
- The ReAct answer is correct and cited but doesn't say reasoning and acting are *interleaved*. Left as a known partial after the fix attempt broke other questions.
- `retrieval_hit` only checks that the right paper was retrieved, not the right passage.

## Next steps

1. Hybrid search (BM25 plus embeddings) and chunk-size experiments.
2. Replace hand grading with an LLM judge, and measure how often it agrees with human grades (Project 2).
3. Grow the test set and re-tune the gate threshold.